# OneDrive CSV sync production audit

## tl;dr

The sample telemetry row and all three schema definitions agree on 71 columns, but the automation is **not ready for unattended production**. The highest-impact gaps are stale data after a header-only run reset, unreliable scheduled-task configuration when the documented relative path is used, successful process exit despite per-account failures, and incomplete row/file integrity checks.

## Context & Methods

This is a bounded data-quality and static contract audit of the repository's MT4 CSV → PowerShell → local OneDrive → Power Query path. The intended grain is one row per closed basket. The candidate analytical key is `AccountNumber + RunID + BasketID`.

### Key Assumptions

- `accounts.csv` is maintained by a trusted VPS operator.
- OneDrive synchronization runs under the same Windows user as the scheduled task.
- The single fixture row is a contract sample, not production history; freshness, duplicate rates over time, and live cloud delivery cannot be measured from it.
- PowerShell runtime tests require Windows/PowerShell and are reported separately when unavailable.

In [ ]:
from pathlib import Path
import csv
import re
from datetime import datetime

repo_root = next(
    candidate for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / 'automation' / 'MoneyMachineCsvSync').exists()
)
fixture_path = repo_root / 'tests' / 'fixtures' / 'AGOLD___Baskets_v3.csv'
sync_path = repo_root / 'automation' / 'MoneyMachineCsvSync' / 'Sync-BasketsToOneDrive.ps1'
installer_path = repo_root / 'automation' / 'MoneyMachineCsvSync' / 'Install-BasketsSyncTask.ps1'
power_query_path = repo_root / 'automation' / 'MoneyMachineCsvSync' / 'PowerQuery' / 'MoneyMachine_Baskets.m'
writer_path = repo_root / 'AmmarTradingGoldEA - ref reset every bar - V3.mq4'

fixture_rows = list(csv.DictReader(fixture_path.open(encoding='utf-8-sig', newline='')))
fixture_header = list(fixture_rows[0].keys())
power_query_source = power_query_path.read_text(encoding='utf-8')
writer_source = writer_path.read_text(encoding='utf-8')
sync_source = sync_path.read_text(encoding='utf-8')
installer_source = installer_path.read_text(encoding='utf-8')

expected_block = re.search(r'ExpectedColumns\s*=\s*\{(.*?)\}\s*,\s*Files', power_query_source, re.S)
power_query_columns = re.findall(r'"([^"]+)"', expected_block.group(1))
writer_header = re.search(r'return "(AccountNumber,.*?,CsvSchemaVersion)";', writer_source).group(1).split(',')

print(f'Repository root: {repo_root}')
print(f'Fixture rows: {len(fixture_rows)}; columns: {len(fixture_header)}')

## Data

The checks below reconcile the fixture, Power Query projection, and MQL4 writer header, then profile the one supplied basket row against the stable business rules that can be tested from this repository.

In [ ]:
sample = fixture_rows[0]
start_time = datetime.strptime(sample['StartTime'], '%Y-%m-%d %H:%M:%S')
end_time = datetime.strptime(sample['EndTime'], '%Y-%m-%d %H:%M:%S')
required_columns = ['AccountNumber', 'BasketID', 'ClosePL', 'CsvSchemaVersion']
candidate_key = (sample['AccountNumber'], sample['RunID'], sample['BasketID'])

quality_checks = {
    'fixture_vs_power_query_exact_order': fixture_header == power_query_columns,
    'fixture_vs_writer_exact_order': fixture_header == writer_header,
    'all_required_values_present': all(sample[column].strip() for column in required_columns),
    'schema_version_is_3': sample['CsvSchemaVersion'] == '3',
    'duration_matches_timestamps': int(sample['DurationSeconds']) == int((end_time - start_time).total_seconds()),
    'trade_date_matches_start': sample['TradeDate'] == sample['StartTime'][:10],
    'direction_allowed': sample['Direction'] in {'BUY', 'SELL'},
    'close_reason_allowed': sample['CloseReason'] in {'TP', 'KILL', 'OTHER'},
    'outcome_allowed': sample['OutcomeClass'] in {'KILL', 'NORMAL_PROFIT', 'RECOVERY_PROFIT', 'OTHER'},
}

print('Candidate key:', candidate_key)
for check, passed in quality_checks.items():
    print(f'{check}: {passed}')

## Results

The schema contract and supplied row are internally consistent. That is necessary but insufficient for production: the sync validator checks only four header names and the first row's account, and the current test intentionally appends a non-CSV comment line that is still copied successfully.

In [ ]:
configuration_fields = [
    'FixedLots', 'PipsStep', 'TakeProfit', 'KillEquityLevel', 'MaxOrdersInBasket',
    'MaxTotalLotsInBasket', 'Magic', 'PointsPerPip', 'Tral', 'TralStart',
    'MaxSpread', 'TimeStart', 'TimeEnd', 'OpenTime', 'NewBasketDelaySeconds',
    'SpeedEA', 'UseBasketTrailingTP', 'TrailingStart', 'TrailingStep',
    'KillSwitchEnable', 'KillCooldownMinutes', 'RegimeEnable', 'RegimeAction',
    'RegimeADXPeriod', 'RegimeADXLevel', 'RegimeADXBars', 'RegimeRangeBars',
    'RegimeRecoveryBars', 'EnableTradingDaysFilter', 'TradeMonday', 'TradeTuesday',
    'TradeWednesday', 'TradeThursday', 'TradeFriday', 'EnableRecoveryStepUp',
    'RecoveryWaitMinutes', 'RecoveryMaxTotalLotsInBasket',
]
fingerprint_block = re.search(r'WithConfigFingerprint.*?List.Transform\(\{(.*?)\}', power_query_source, re.S).group(1)
fingerprint_fields = re.findall(r'\[([^\]]+)\]', fingerprint_block)
missing_fingerprint_fields = [field for field in configuration_fields if field not in fingerprint_fields]

static_evidence = {
    'header_only_file_is_rejected': 'CSV contains no basket row with AccountNumber' in sync_source,
    'post_copy_check_uses_zero_second_fresh_snapshot': 'Test-StableFile -Path $sourceCsv -Seconds 0' in sync_source,
    'installer_preserves_config_argument_without_resolving': 'Resolve-Path' not in installer_source and '$quotedConfig' in installer_source,
    'installer_creates_two_independent_tasks': '$TaskName-Daily' in installer_source and '$TaskName-StartupCatchup' in installer_source,
    'sync_has_explicit_process_exit': bool(re.search(r'(?im)^\s*exit\s+\d+', sync_source)),
}

print(f'Fingerprint fields: {len(fingerprint_fields)} of {len(configuration_fields)}')
print('Missing fingerprint fields:', ', '.join(missing_fingerprint_fields))
for check, observed in static_evidence.items():
    print(f'{check}: {observed}')

## Takeaways

- **Verified:** the fixture, MQL4 writer, and Power Query projection have the same 71 columns in the same order; the one sample row passes the available cross-field checks.
- **Blocking:** the static automation contract does not guarantee freshness, complete CSV validity, cloud delivery, or observable failure.
- **Coverage gap:** only one sample basket exists, no real `sync.log`/state history was supplied, and this Linux environment has no PowerShell runtime. Windows Task Scheduler, OneDrive upload, file-lock, locale, and recovery behavior require an acceptance test on the target VPS.
- **Decision:** fix the blocking behaviors and pass the Windows acceptance matrix before unattended production use.